# 03d. Enriquecimiento de tracks con información de artistas

Este notebook une la tabla `tracks_scored_v2` con la tabla `artists_ref`, construida previamente a partir de `artists.csv`.

El objetivo es agregar variables a nivel artista para cada canción, principalmente:

- `max_artist_popularity`
- `max_artist_followers`
- `artist_names_ref`
- `num_artist_ids_in_track`
- `num_artists_joined`

Esta etapa es necesaria antes de construir `hidden_gems_v3`, porque la popularidad baja de un track no garantiza que el artista sea poco conocido. Una canción puede tener baja popularidad en una fila específica del dataset, pero pertenecer a un artista globalmente famoso.

Por ello, antes de aplicar filtros finales, se diagnostica la calidad del join entre tracks y artistas.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, StringType

In [2]:
spark = (
    SparkSession.builder
    .appName("03d_tracks_artist_enrichment")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/21 03:39:07 WARN Utils: Your hostname, debian, resolves to a loopback address: 127.0.1.1; using 192.168.100.25 instead (on interface wlp3s0)
26/05/21 03:39:07 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/21 03:39:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
PROCESSED_DIR = "../processed"

TRACKS_SCORED_PATH = f"{PROCESSED_DIR}/tracks_scored.parquet"
ARTISTS_REF_PATH = f"{PROCESSED_DIR}/artists_ref.parquet"

TRACKS_ENRICHED_ARTISTS_PATH = f"{PROCESSED_DIR}/tracks_enriched_artists.parquet"

DIAG_DIR = f"{PROCESSED_DIR}/diagnostics/tracks_artist_enrichment"

## Carga de tablas

Se cargan dos tablas:

1. `tracks_scored_v2`: contiene las canciones ya limpiadas y con scores acústicos.
2. `artists_ref`: contiene la información limpia y clasificada de artistas.

Antes de unirlas, se revisan los schemas para identificar correctamente las columnas de IDs de tracks y artistas.

In [4]:
tracks_scored = spark.read.parquet(TRACKS_SCORED_PATH).cache()
artists_ref = spark.read.parquet(ARTISTS_REF_PATH).cache()

26/05/21 03:39:14 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [5]:
print(f"Filas en tracks_scored: {tracks_scored.count():,}")
print(f"Filas en artists_ref: {artists_ref.count():,}")

Filas en tracks_scored: 586,264


[Stage 6:=======>                                                   (1 + 7) / 8]

Filas en artists_ref: 1,162,095


In [6]:
tracks_scored.printSchema()

root
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- artists: string (nullable = true)
 |-- id_artists: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- release_decade: integer (nullable = true)
 |-- duration_ms: integer (nullable = true)
 |-- duration_min: double (nullable = true)
 |-- explicit: integer (nullable = true)
 |-- popularity: integer (nullable = true)
 |-- popularity_class: string (nullable = true)
 |-- is_hit: integer (nullable = true)
 |-- danceability: double (nullable = true)
 |-- energy: double (nullable = true)
 |-- key: integer (nullable = true)
 |-- loudness: double (nullable = true)
 |-- mode: integer (nullable = true)
 |-- speechiness: double (nullable = true)
 |-- acousticness: double (nullable = true)
 |-- instrumentalness: double (nullable = true)
 |-- liveness: double (nullable = true)
 |-- valence: double (nullable = true)
 |-- tempo: double (nullable = true)
 |-

In [7]:
[c for c in tracks_scored.columns if "artist" in c.lower()]

['artists', 'id_artists']

## Revisión de artistas en tracks

El schema de `tracks_scored` muestra que cada canción conserva dos columnas relevantes para artistas:

- `artists`: nombres de artistas como texto.
- `id_artists`: identificadores de artistas asociados al track.

Para unir con `artists_ref`, se necesita usar `id_artists`, ya que `artists_ref` está indexado por `artist_id`. Antes de hacer el join, se revisa cómo viene almacenada esta columna, ya que normalmente aparece como una lista serializada en texto.

In [9]:
#ver muestra enfocada
tracks_scored.select(
    "id",
    "name",
    "artists",
    "id_artists",
    "popularity",
    "release_year",
    "hit_score"
).show(20, truncate=False, vertical=True)

-RECORD 0--------------------------------------------------------------------------------------
 id           | 4pCfNA9bjI3pccG5s9fm7J                                                         
 name         | Pumping On Your Stereo                                                         
 artists      | ['Supergrass']                                                                 
 id_artists   | ['0sHeX8oQ6o7xic3wMf4NBU']                                                     
 popularity   | 50                                                                             
 release_year | 1999                                                                           
 hit_score    | 81.48                                                                          
-RECORD 1--------------------------------------------------------------------------------------
 id           | 6Gk3mU3np8miLUtZjoIVCu                                                         
 name         | Mujhe Raat Din Bas      

## Parseo de IDs de artistas

La columna `id_artists` contiene los identificadores de artistas asociados a cada track, pero está almacenada como texto. Para poder unir tracks con `artists_ref`, primero se transforma esta columna en un arreglo real de Spark.

Este paso es necesario porque algunas canciones tienen más de un artista. Después del parseo, se podrá aplicar `explode` para crear una fila por combinación track-artista.

In [10]:
empty_string_array = F.array().cast(ArrayType(StringType()))

id_artists_inner = F.regexp_replace(
    F.regexp_replace(F.col("id_artists"), r"^\[", ""),
    r"\]$",
    ""
)

id_artists_split = F.split(id_artists_inner, r",\s*")

id_artists_cleaned = F.transform(
    id_artists_split,
    lambda x: F.regexp_replace(
        F.trim(x),
        r"""^['"]|['"]$""",
        ""
    )
)

tracks_with_artist_ids = (
    tracks_scored
    .withColumnRenamed("id", "track_id")
    .withColumnRenamed("name", "track_name")
    .withColumn(
        "artist_ids",
        F.when(
            (F.col("id_artists").isNull()) | (F.col("id_artists") == "[]"),
            empty_string_array
        ).otherwise(id_artists_cleaned)
    )
    .withColumn(
        "artist_ids",
        F.expr("filter(artist_ids, x -> x is not null and trim(x) <> '')")
    )
    .withColumn("num_artist_ids_in_track", F.size(F.col("artist_ids")))
    .cache()
)

In [12]:
tracks_with_artist_ids.select(
    "track_id",
    "track_name",
    "artists",
    "id_artists",
    "artist_ids",
    "num_artist_ids_in_track"
).show(20, truncate=False, vertical=True)

-RECORD 0-------------------------------------------------------------------------------------------------
 track_id                | 4pCfNA9bjI3pccG5s9fm7J                                                         
 track_name              | Pumping On Your Stereo                                                         
 artists                 | ['Supergrass']                                                                 
 id_artists              | ['0sHeX8oQ6o7xic3wMf4NBU']                                                     
 artist_ids              | [0sHeX8oQ6o7xic3wMf4NBU]                                                       
 num_artist_ids_in_track | 1                                                                              
-RECORD 1-------------------------------------------------------------------------------------------------
 track_id                | 6Gk3mU3np8miLUtZjoIVCu                                                         
 track_name              | Mujhe Raat

In [13]:
artist_ids_summary = tracks_with_artist_ids.select(
    F.count("*").alias("n_tracks"),
    F.sum((F.col("num_artist_ids_in_track") == 0).cast("int")).alias("tracks_without_artist_ids"),
    F.sum((F.col("num_artist_ids_in_track") > 0).cast("int")).alias("tracks_with_artist_ids"),
    F.avg("num_artist_ids_in_track").alias("avg_artist_ids_per_track"),
    F.max("num_artist_ids_in_track").alias("max_artist_ids_per_track")
)

artist_ids_summary.show(truncate=False)

[Stage 14:====================================>                     (5 + 3) / 8]

+--------+-------------------------+----------------------+------------------------+------------------------+
|n_tracks|tracks_without_artist_ids|tracks_with_artist_ids|avg_artist_ids_per_track|max_artist_ids_per_track|
+--------+-------------------------+----------------------+------------------------+------------------------+
|586264  |0                        |586264                |1.2904681167528622      |58                      |
+--------+-------------------------+----------------------+------------------------+------------------------+



## Explode de artistas por track

Después de convertir `id_artists` a un arreglo real, se aplica `explode_outer` para crear una fila por cada combinación track-artista.

Esto permite unir cada artista individual con `artists_ref`. Posteriormente, la información se agregará de nuevo a nivel track usando métricas como el máximo de popularidad y seguidores entre los artistas participantes.

In [14]:
tracks_artists_exploded = (
    tracks_with_artist_ids
    .withColumn("artist_id", F.explode_outer("artist_ids"))
    .cache()
)

In [15]:
print(f"Tracks originales: {tracks_with_artist_ids.count():,}")
print(f"Filas track-artista después de explode: {tracks_artists_exploded.count():,}")

Tracks originales: 586,264


[Stage 21:=======>                                                  (1 + 7) / 8]

Filas track-artista después de explode: 756,555


### Resultado del explode

El proceso de `explode` transformó la tabla de tracks de una fila por canción a una fila por combinación track-artista.

La tabla original tiene 586,264 canciones, mientras que la tabla expandida tiene 756,555 pares track-artista. Esto es consistente con el promedio observado de aproximadamente 1.29 artistas por canción.

Esta representación intermedia permite unir cada artista individual con `artists_ref`.

In [16]:
#Diagnostico formal de explode
explode_summary = tracks_artists_exploded.select(
    F.count("*").alias("n_track_artist_rows"),
    F.countDistinct("track_id").alias("n_unique_tracks"),
    F.countDistinct("artist_id").alias("n_unique_artist_ids_in_tracks"),
    F.sum(F.col("artist_id").isNull().cast("int")).alias("null_artist_id_rows")
)

explode_summary.show(truncate=False)

[Stage 25:>                                                         (0 + 8) / 8]

+-------------------+---------------+-----------------------------+-------------------+
|n_track_artist_rows|n_unique_tracks|n_unique_artist_ids_in_tracks|null_artist_id_rows|
+-------------------+---------------+-----------------------------+-------------------+
|756555             |586264         |98400                        |0                  |
+-------------------+---------------+-----------------------------+-------------------+



### Diagnóstico del explode

El diagnóstico confirma que el proceso de expansión fue correcto. La tabla expandida contiene 756,555 pares track-artista y conserva los 586,264 tracks originales.

Además, no se generaron filas con `artist_id` nulo. Esto indica que el parseo de `id_artists` y el posterior `explode` son confiables para continuar con el join contra `artists_ref`.

In [17]:
tracks_artists_joined = (
    tracks_artists_exploded
    .join(
        artists_ref,
        on="artist_id",
        how="left"
    )
    .cache()
)

In [18]:
tracks_artists_joined.select(
    "track_id",
    "track_name",
    "artist_id",
    "artist_name",
    "artist_popularity",
    "artist_followers",
    "artist_popularity_class",
    "artist_size_class"
).show(30, truncate=False, vertical=True)

[Stage 35:====================================================> (194 + 6) / 200]

-RECORD 0-------------------------------------------------------------
 track_id                | 0aTltS2EBEW76JBVQhIQ7a                     
 track_name              | Lonely-lonely                              
 artist_id               | 00hP5aJk3HbvFpXzcBeSl3                     
 artist_name             | The Narrow                                 
 artist_popularity       | 20                                         
 artist_followers        | 6230.0                                     
 artist_popularity_class | baja                                       
 artist_size_class       | pequeño                                    
-RECORD 1-------------------------------------------------------------
 track_id                | 5kkZl3e7K1WwxTdwo9t3QJ                     
 track_name              | Almendra                                   
 artist_id               | 00n4Vljc6N9pvJ26SKPphh                     
 artist_name             | Ruben Gonzalez                             
 artis

In [19]:
join_quality_summary = tracks_artists_joined.select(
    F.count("*").alias("n_track_artist_rows"),
    F.sum(F.col("artist_name").isNotNull().cast("int")).alias("matched_artist_rows"),
    F.sum(F.col("artist_name").isNull().cast("int")).alias("unmatched_artist_rows"),
    F.avg(F.col("artist_name").isNotNull().cast("double")).alias("match_rate_rows")
)

join_quality_summary.show(truncate=False)

+-------------------+-------------------+---------------------+------------------+
|n_track_artist_rows|matched_artist_rows|unmatched_artist_rows|match_rate_rows   |
+-------------------+-------------------+---------------------+------------------+
|756555             |730529             |26026                |0.9655993285352684|
+-------------------+-------------------+---------------------+------------------+



In [20]:
join_quality_by_track = (
    tracks_artists_joined
    .groupBy("track_id")
    .agg(
        F.count("*").alias("n_artist_rows"),
        F.sum(F.col("artist_name").isNotNull().cast("int")).alias("n_artists_matched"),
        F.sum(F.col("artist_name").isNull().cast("int")).alias("n_artists_unmatched")
    )
)

In [21]:
track_join_summary = join_quality_by_track.select(
    F.count("*").alias("n_tracks"),
    F.sum((F.col("n_artists_matched") > 0).cast("int")).alias("tracks_with_at_least_one_artist_match"),
    F.sum((F.col("n_artists_matched") == 0).cast("int")).alias("tracks_with_no_artist_match"),
    F.avg((F.col("n_artists_matched") > 0).cast("double")).alias("track_match_rate")
)

track_join_summary.show(truncate=False)

[Stage 48:==================================================>   (187 + 8) / 200]

+--------+-------------------------------------+---------------------------+------------------+
|n_tracks|tracks_with_at_least_one_artist_match|tracks_with_no_artist_match|track_match_rate  |
+--------+-------------------------------------+---------------------------+------------------+
|586264  |575848                               |10416                      |0.9822332601012513|
+--------+-------------------------------------+---------------------------+------------------+



Artistas sin match frecuentes:

In [25]:
unmatched_artist_ids = (
    tracks_artists_joined
    .filter(F.col("artist_name").isNull())
    .groupBy("artist_id")
    .agg(
        F.count("*").alias("n_track_appearances"),
        F.first("track_name", ignorenulls=True).alias("example_track"),
        F.first("artists", ignorenulls=True).alias("artists_raw_example"),
        F.first("id_artists", ignorenulls=True).alias("id_artists_raw_example")
    )
    .orderBy(F.desc("n_track_appearances"))
)

unmatched_artist_ids.show(30, truncate=False, vertical=True)

-RECORD 0------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 artist_id              | 0LyfQWJT6nXafLPZqxe9Of                                                                                                                                                                                                                                                                                                   
 n_track_appearances    | 321                                                                                                                                                                                                                                                                                                   

Tracks sin ningún artista encontrado

In [27]:
tracks_with_no_artist_match = (
    join_quality_by_track
    .filter(F.col("n_artists_matched") == 0)
    .join(
        tracks_with_artist_ids.select(
            "track_id",
            "track_name",
            "artists",
            "id_artists",
            "artist_ids",
            "popularity",
            "release_year",
            "hit_score"
        ),
        on="track_id",
        how="left"
    )
)

tracks_with_no_artist_match.show(30, truncate=False, vertical= True)

[Stage 82:====================================>                     (5 + 3) / 8]

-RECORD 0-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 track_id            | 04fb417rO1LK0YhrymR7Ib                                                                                                                                                                 
 n_artist_rows       | 1                                                                                                                                                                                      
 n_artists_matched   | 0                                                                                                                                                                                      
 n_artists_unmatched | 1                                                                                                                                                    

In [28]:
#Crear Features agregadas por track
artist_features_by_track = (
    tracks_artists_joined
    .groupBy("track_id")
    .agg(
        F.max("artist_popularity").alias("max_artist_popularity"),
        F.max("artist_followers").alias("max_artist_followers"),

        F.collect_set("artist_name").alias("artist_names_ref"),
        F.collect_set("artist_popularity_class").alias("artist_popularity_classes"),
        F.collect_set("artist_size_class").alias("artist_size_classes"),

        F.countDistinct("artist_id").alias("num_artist_ids_exploded"),
        F.countDistinct(
            F.when(F.col("artist_name").isNotNull(), F.col("artist_id"))
        ).alias("num_artists_joined")
    )
    .cache()
)

In [29]:
#validar conteo
print(f"Filas en artist_features_by_track: {artist_features_by_track.count():,}")
print(f"Tracks únicos: {artist_features_by_track.select('track_id').distinct().count():,}")

Filas en artist_features_by_track: 586,264
Tracks únicos: 586,264


In [31]:
#unir con tracks originales
tracks_enriched_artists = (
    tracks_with_artist_ids
    .join(
        artist_features_by_track,
        on="track_id",
        how="left"
    )
    .cache()
)

26/05/21 04:04:07 WARN CacheManager: Asked to cache already cached data.


In [32]:
print(f"Filas en tracks_with_artist_ids: {tracks_with_artist_ids.count():,}")
print(f"Filas en tracks_enriched_artists: {tracks_enriched_artists.count():,}")

Filas en tracks_with_artist_ids: 586,264


[Stage 131:==================================================>  (192 + 8) / 200]

Filas en tracks_enriched_artists: 586,264


In [33]:
#Diagnostico Final del enrequecimiento 
tracks_enriched_summary = tracks_enriched_artists.select(
    F.count("*").alias("n_tracks"),

    F.sum(F.col("max_artist_popularity").isNotNull().cast("int")).alias("tracks_with_artist_popularity"),
    F.sum(F.col("max_artist_popularity").isNull().cast("int")).alias("tracks_without_artist_popularity"),

    F.sum(F.col("max_artist_followers").isNotNull().cast("int")).alias("tracks_with_artist_followers"),
    F.sum(F.col("max_artist_followers").isNull().cast("int")).alias("tracks_without_artist_followers")
)

tracks_enriched_summary.show(truncate=False)

+--------+-----------------------------+--------------------------------+----------------------------+-------------------------------+
|n_tracks|tracks_with_artist_popularity|tracks_without_artist_popularity|tracks_with_artist_followers|tracks_without_artist_followers|
+--------+-----------------------------+--------------------------------+----------------------------+-------------------------------+
|586264  |575848                       |10416                           |575848                      |10416                          |
+--------+-----------------------------+--------------------------------+----------------------------+-------------------------------+



In [35]:
#Muestra de tracks enriquecido
tracks_enriched_artists.select(
    "track_id",
    "track_name",
    "artists",
    "artist_names_ref",
    "popularity",
    "release_year",
    "hit_score",
    "max_artist_popularity",
    "max_artist_followers",
    "artist_popularity_classes",
    "artist_size_classes",
    "num_artist_ids_in_track",
    "num_artists_joined"
).orderBy(F.desc("max_artist_followers")).show(30, truncate=False, vertical=True)

-RECORD 0-----------------------------------------------------------------------------
 track_id                  | 2z7dSOS87BhVTTGOopZhC5                                   
 track_name                | I Was Made For Loving You                                
 artists                   | ['Tori Kelly', 'Ed Sheeran']                             
 artist_names_ref          | [Ed Sheeran, Tori Kelly]                                 
 popularity                | 54                                                       
 release_year              | 2016                                                     
 hit_score                 | 60.43                                                    
 max_artist_popularity     | 92                                                       
 max_artist_followers      | 7.8900234E7                                              
 artist_popularity_classes | [alta]                                                   
 artist_size_classes       | [masivo]      

## Guardar `tracks_enriched_artists.parquet`

In [36]:
TRACKS_ENRICHED_ARTISTS_PATH = "../processed/tracks_enriched_artists.parquet"

tracks_enriched_artists.write.mode("overwrite").parquet(
    TRACKS_ENRICHED_ARTISTS_PATH
)

26/05/21 04:09:41 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/05/21 04:09:41 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/05/21 04:09:42 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/05/21 04:09:43 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

In [37]:
tracks_enriched_check = spark.read.parquet(TRACKS_ENRICHED_ARTISTS_PATH)

print(f"Filas guardadas en tracks_enriched_artists: {tracks_enriched_check.count():,}")
tracks_enriched_check.printSchema()

Filas guardadas en tracks_enriched_artists: 586,264
root
 |-- track_id: string (nullable = true)
 |-- track_name: string (nullable = true)
 |-- artists: string (nullable = true)
 |-- id_artists: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- release_decade: integer (nullable = true)
 |-- duration_ms: integer (nullable = true)
 |-- duration_min: double (nullable = true)
 |-- explicit: integer (nullable = true)
 |-- popularity: integer (nullable = true)
 |-- popularity_class: string (nullable = true)
 |-- is_hit: integer (nullable = true)
 |-- danceability: double (nullable = true)
 |-- energy: double (nullable = true)
 |-- key: integer (nullable = true)
 |-- loudness: double (nullable = true)
 |-- mode: integer (nullable = true)
 |-- speechiness: double (nullable = true)
 |-- acousticness: double (nullable = true)
 |-- instrumentalness: double (nullable = true)
 |-- liveness: double (nullable = true)
 |-- valence: d

## Conclusión del enriquecimiento

El enriquecimiento de tracks con información de artistas fue exitoso.

La tabla final conserva los 586,264 tracks originales y agrega variables a nivel artista para 575,848 canciones, lo que representa aproximadamente el 98.2% del total. Los 10,416 tracks restantes no encontraron coincidencia en `artists_ref`, por lo que sus variables de artista permanecen como nulas.

Las variables agregadas más importantes son:

- `max_artist_popularity`
- `max_artist_followers`
- `artist_names_ref`
- `artist_popularity_classes`
- `artist_size_classes`
- `num_artist_ids_in_track`
- `num_artists_joined`

Se utiliza el máximo de popularidad y seguidores porque una canción con múltiples artistas no debería considerarse hidden gem estricta si al menos uno de sus artistas es masivo.

La siguiente etapa será construir `hidden_gems_v3`, usando tanto variables de track como variables de artista.